# Reinforcement Learning (Q-Learning)


## ¿Qué es el Reinforcement Learning?

El Reinforcement Learning es como enseñar mediante ensayo y error: un Agent aprende probando acciones, recibiendo retroalimentación (Rewards) y mejorando gradualmente su comportamiento. Piensa en entrenar a una mascota con premios, aprender a andar en bicicleta con práctica, o dominar un videojuego jugándolo repetidamente.


En esencia, el RL trata de enseñar a un **Agent** (nuestro algoritmo) cómo comportarse en un **Environment** (nuestro juego o problema) para maximizar un **Reward**.

Se basa en un ciclo simple:
1.  El **Agent** observa el **State** (S) del Environment.
2.  El **Agent** elige una **Action** (A).
3.  El **Environment** reacciona: le da al Agent un **Reward** (R) y un nuevo **State** (S').
4.  El Agent aprende de esta tupla (S, A, R, S') y el ciclo se repite.

Nuestro objetivo es crear un Agent que, desde cualquier State, aprenda a elegir la Action que le dará el mayor *Reward futuro acumulado*.


[![Reinforcement Learning Diagram](../img/RL_loop.png)](https://gymnasium.farama.org/introduction/basic_usage/)


## Entendiendo Q-Learning Intuitivamente

Q-learning construye una gran "hoja de trucos" llamada Q-table que le dice al Agent qué tan buena es cada Action en cada situación:
- Filas = diferentes situaciones (States) que el Agent puede encontrar
- Columnas = diferentes Actions que el Agent puede tomar
- Valores = qué tan buena es esa Action en esa situación (Reward futuro esperado)

### El Proceso de Aprendizaje

1. Probar una Action y ver qué pasa (Reward + nuevo State)
2. Actualizar la hoja de trucos: "Esa Action fue mejor/peor de lo que pensaba"
3. Mejorar gradualmente probando Actions y actualizando estimaciones
4. Equilibrar exploración vs explotación: Probar cosas nuevas vs usar lo que ya sabes que funciona

Por qué funciona: Con el tiempo, las buenas Actions obtienen Q-values más altos, las malas Actions obtienen Q-values más bajos. El Agent aprende a elegir Actions con los Rewards esperados más altos.


<img src="https://huggingface.co/blog/assets/70_deep_rl_q_part1/Q-function-2.jpg" alt="Q function"/>

In [ ]:
import numpy as np
import random
import time
from IPython.display import clear_output

### Environment

Crearemos un Grid World simple de 4x4.

* **Agent (A):** Nuestro aprendiz.
* **Goal (G):** Un State bueno con un Reward de +10.
* **Trap (T):** Un State malo con un Reward de -10.
* **States (S):** Cualquier posición en la cuadrícula, representada como `(fila, columna)`.
* **Actions (A):** Arriba, Abajo, Izquierda, Derecha.
* **Rewards (R):**
    * -0.1 por cada paso (para fomentar la velocidad).
    * +10 al alcanzar el Goal.
    * -10 al caer en el Trap.

Aquí está nuestro mapa:
```
  (0,0) (0,1) (0,2) (0,3)
  (1,0) (1,1) (1,2) (1,3)
  (2,0) (2,1) [T]   (2,3)
  (3,0) (3,1) (3,2) [G]
```

In [ ]:
# Parámetros del Environment
GRID_ROWS = 4
GRID_COLS = 4
START_STATE = (0, 0)
GOAL_STATE = (3, 3)
TRAP_STATE = (2, 2)

# Rewards
REWARD_STEP = -0.1
REWARD_GOAL = 10
REWARD_TRAP = -10

# Actions (0: Arriba, 1: Abajo, 2: Izquierda, 3: Derecha)
# Usamos índices para facilitar la búsqueda en nuestra Q-table
ACTIONS = [0, 1, 2, 3]
ACTION_NAMES = ["↑", "↓", "←", "→"]

# Un diccionario para mapear índices de Actions a (cambio_fila, cambio_columna)
# Así es como nos moveremos en la cuadrícula
ACTION_VECTORS = {
    0: (-1, 0), # Arriba
    1: (1, 0),  # Abajo
    2: (0, -1), # Izquierda
    3: (0, 1)   # Derecha
}

print(f"Grid World: {GRID_ROWS}x{GRID_COLS}")
print(f"Inicio: {START_STATE}, Goal: {GOAL_STATE}, Trap: {TRAP_STATE}")
print(f"Actions: {list(zip(ACTIONS, ACTION_NAMES))}")

### El Cerebro del Agent (La Q-Table)

¿Cómo "aprenderá" nuestro Agent? Almacenará su conocimiento en una **Q-Table**.

* Es una gran tabla (un array NumPy 3D en nuestro caso) donde:
    * Las filas representan las **filas** de la cuadrícula.
    * Las columnas representan las **columnas** de la cuadrícula.
    * La 3ª dimensión representa la **Action**.

* `Q_table[fila, columna, action]` almacenará un número, el "Q-value".
* Este **Q-value** es la *predicción* del Agent del Reward futuro total que obtendrá si toma esa `action` desde ese State `(fila, columna)`.

Inicializamos esta tabla con todos ceros, porque nuestro Agent empieza sin saber nada.

In [ ]:
Q_table = np.zeros((GRID_ROWS, GRID_COLS, len(ACTIONS)))

print("Q-Table inicial (todos ceros):")
print(Q_table)
print(f"Forma de la Q-Table: {Q_table.shape}")

### El Algoritmo de Aprendizaje (Q-Learning)

Este es el concepto más importante. ¿Cómo actualizamos la Q-table?

Cuando el Agent toma una **Action** ($a$) desde un **State** ($s$), se mueve a un **nuevo State** ($s'$) y obtiene un **Reward** ($r$), actualizamos la tabla usando la **Bellman Equation**:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

Desglosemos esto en términos simples:

* $Q(s, a) \leftarrow ...$
    * "Actualiza el Q-value para el State y la Action anteriores..."
* $... + \alpha [...]$ 
    * "...añadiendo una pequeña porción (la **tasa de aprendizaje**, $\alpha$) de nueva información."
* $... [r + \gamma \max_{a'} Q(s', a') ...]$
    * Esta es la "nueva información". Es el **Reward ($r$) que acabamos de obtener**...
    * ...más el **mejor Q-value posible que podemos obtener desde nuestro nuevo State ($s'$)**, (multiplicado por un **factor de descuento**, $\gamma$, que valora los Rewards inmediatos sobre los futuros).
* $... - Q(s, a)]$
    * Todo el término entre corchetes `[ ... ]` es el "error de diferencia temporal": la diferencia entre nuestra *nueva estimación* ($r + \gamma \max...$) y nuestra *estimación anterior* ($Q(s, a)$).

También necesitamos una **estrategia** para elegir Actions. Usaremos **Epsilon-Greedy ($\epsilon$-greedy)**:

* Con probabilidad `1 - epsilon`: **Explotación** (elige la mejor Action conocida de la Q-table).
* Con probabilidad `epsilon`: **Exploración** (elige una Action aleatoria para descubrir nuevos caminos).

Esto asegura que nuestro Agent no quede atascado en el primer camino "bueno" que encuentre.

In [ ]:
# Hiperparámetros
learning_rate = 0.1   # Alpha (α): Qué tan rápido aprende el Agent.
discount_factor = 0.9 # Gamma (γ): Cuánto valora el Agent los Rewards futuros.
epsilon = 1.0         # Tasa de exploración inicial
max_epsilon = 1.0     # Exploración máxima
min_epsilon = 0.01    # Exploración mínima
epsilon_decay = 0.001 # Tasa a la que disminuye la exploración

# Parámetros de entrenamiento
total_episodes = 10000 # Cuántos "juegos" jugar
max_steps = 100        # Pasos máximos por juego (para evitar bucles infinitos)

### Funciones Auxiliares

Necesitamos algunas funciones para ejecutar nuestra simulación.

1.  `choose_action(state)`: Implementa la estrategia $\epsilon$-greedy.
2.  `take_action(state, action_index)`: Simula tomar una Action y devuelve la tupla `(nuevo_state, reward, done)`. Esta función también manejará "chocar con paredes" (quedarse en el mismo State).

In [ ]:
def choose_action(state, current_epsilon):
    """
    Elige una Action usando la estrategia Epsilon-Greedy.
    """
    row, col = state
    
    # Decisión Epsilon-Greedy
    if random.uniform(0, 1) < current_epsilon:
        # Explorar: elegir una Action aleatoria
        return random.choice(ACTIONS)
    else:
        # Explotar: elegir la mejor Action de la Q-table
        # np.argmax encuentra el índice (0, 1, 2 o 3) del Q-value más alto
        return np.argmax(Q_table[row, col])

def take_action(state, action_index):
    """
    Toma una Action, calcula el nuevo State, el Reward, y si el episodio terminó.
    """
    current_row, current_col = state
    action_row, action_col = ACTION_VECTORS[action_index]
    
    # Calcular nueva posición potencial
    new_row = current_row + action_row
    new_col = current_col + action_col
    
    # --- Verificar colisiones con paredes ---
    # Limitar la fila al rango [0, GRID_ROWS - 1]
    new_row = max(0, min(new_row, GRID_ROWS - 1))
    # Limitar la columna al rango [0, GRID_COLS - 1]
    new_col = max(0, min(new_col, GRID_COLS - 1))
    
    new_state = (new_row, new_col)
    
    # --- Obtener Reward y verificar si terminó ---
    if new_state == GOAL_STATE:
        reward = REWARD_GOAL
        done = True
    elif new_state == TRAP_STATE:
        reward = REWARD_TRAP
        done = True
    else:
        reward = REWARD_STEP
        done = False
        
    return new_state, reward, done

### El Bucle de Entrenamiento

¡Aquí es donde todo se une! Simularemos muchos "episodios" (juegos). En cada episodio, el Agent se mueve paso a paso hasta alcanzar el Goal o caer en el Trap.

En *cada paso individual*, haremos:
1.  Elegir una Action.
2.  Tomar la Action y obtener el resultado `(nuevo_state, reward, done)`.
3.  **Actualizar nuestra Q-table** usando la Bellman Equation.
4.  Moverse al nuevo State.

También "decaeremos" epsilon con el tiempo, para que el Agent explore mucho al principio y luego comience a explotar más su conocimiento a medida que aprende.

In [ ]:
# Para almacenar Rewards y graficarlos después
episode_rewards = []
current_epsilon = epsilon

for episode in range(total_episodes):
    state = START_STATE
    total_reward = 0
    
    for step in range(max_steps):
        # 1. Elegir una Action
        action_index = choose_action(state, current_epsilon)
        
        # 2. Tomar la Action
        new_state, reward, done = take_action(state, action_index)
        
        # 3. Actualizar la Q-table (La fórmula de Q-Learning)
        row, col = state
        new_row, new_col = new_state
        
        old_q_value = Q_table[row, col, action_index]
        
        # Esto es max(Q(s', a')) de la fórmula
        best_future_q = np.max(Q_table[new_row, new_col])
        
        # La regla central de actualización de Q-Learning
        new_q_value = old_q_value + learning_rate * (reward + discount_factor * best_future_q - old_q_value)
        Q_table[row, col, action_index] = new_q_value
        
        # 4. Actualizar State y Reward
        state = new_state
        total_reward += reward
        
        if done:
            break # Episodio terminado
            
    # Después del episodio, almacenar Rewards y decaer epsilon
    episode_rewards.append(total_reward)
    
    # Decaer epsilon
    current_epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-epsilon_decay * episode)
    
    if (episode + 1) % 1000 == 0:
        print(f"Episodio {episode + 1}/{total_episodes} | Epsilon: {current_epsilon:.4f}")

### Resultados

In [ ]:
print("Q-Table Final:\n", Q_table)

Podemos interpretar fácilmente el contenido de la Q-table final como una **Policy**. Para cada celda de la cuadrícula, observaremos sus Q-values y encontraremos la **Action con el Q-value más alto**. Imprimiremos una flecha (`↑`, `↓`, `←`, `→`) para esa Action. ¡Esto nos muestra el *camino óptimo* que el Agent aprendió desde *cualquier* celda!

In [ ]:
print("🎓 Policy Aprendida (Mejor Action desde cada State):")

# Crear una cuadrícula para almacenar las flechas de la Policy
policy_grid = [["" for _ in range(GRID_COLS)] for _ in range(GRID_ROWS)]

for r in range(GRID_ROWS):
    for c in range(GRID_COLS):
        state = (r, c)
        
        if state == GOAL_STATE:
            policy_grid[r][c] = "🏆" # Goal
        elif state == TRAP_STATE:
            policy_grid[r][c] = "🔥" # Trap
        else:
            # Encontrar la mejor Action (índice) desde este State
            best_action_index = np.argmax(Q_table[r, c])
            # Mapear ese índice a su flecha
            policy_grid[r][c] = ACTION_NAMES[best_action_index]

# Imprimir la cuadrícula de Policy
for row in policy_grid:
    # .join(row) combina todos los elementos de la lista en una cadena
    # Usamos \t (tabulador) para espaciarlos bien
    print("\t".join(row))

**Si el entrenamiento fue exitoso, deberías ver un "campo" de flechas apuntando hacia el Goal (🏆) y evitando el Trap (🔥).**

---

### 🎬 Ver al Agent Jugar

Ahora usemos nuestra Policy aprendida (la Q-table) para jugar un juego *sin ninguna exploración* (`epsilon = 0`). Veremos el camino óptimo que aprendió.

In [ ]:
state = START_STATE
total_reward = 0
step_count = 0

for _ in range(max_steps):
    # Imprimir la cuadrícula actual
    clear_output(wait=True) # Limpia la salida para una animación visual
    
    # Crear una cuadrícula temporal para imprimir
    print_grid = [["." for _ in range(GRID_COLS)] for _ in range(GRID_ROWS)]
    print_grid[GOAL_STATE[0]][GOAL_STATE[1]] = "🏆"
    print_grid[TRAP_STATE[0]][TRAP_STATE[1]] = "🔥"
    print_grid[state[0]][state[1]] = "🤖" # Posición actual del Agent
    
    print(f"Paso: {step_count} | Reward Total: {total_reward:.1f}")
    for row in print_grid:
        print("\t".join(row))

    # --- Tomar la MEJOR Action (sin exploración) ---
    row, col = state
    action_index = np.argmax(Q_table[row, col])
    
    new_state, reward, done = take_action(state, action_index)
    
    state = new_state
    total_reward += reward
    step_count += 1
    
    time.sleep(0.5) # Pausa de 0.5 segundos para ver el movimiento
    
    if done:
        # Imprimir el State final
        clear_output(wait=True)
        print_grid = [["." for _ in range(GRID_COLS)] for _ in range(GRID_ROWS)]
        print_grid[GOAL_STATE[0]][GOAL_STATE[1]] = "🏆"
        print_grid[TRAP_STATE[0]][TRAP_STATE[1]] = "🔥"
        print_grid[state[0]][state[1]] = "🤖"
        
        print(f"Paso: {step_count} | Reward Total: {total_reward:.1f}")
        for row in print_grid:
            print("\t".join(row))
        
        if state == GOAL_STATE:
            print("\n🎉 ¡El Agent alcanzó el Goal! 🎉")
        else:
            print("\n☠️ ¡El Agent cayó en el Trap! ☠️")
        break